# 00 · Overview — Credit-Card Fraud Detection

**Goal:** turn a fraud classifier into a decision a manager reads in **dollars**.

We optimize for *expected money saved*, not academic ROC-AUC. The headline metric is net \$ saved at the cost-optimal decision threshold, where:

- A **missed fraud** (false negative) costs the full transaction `Amount`.
- A **false alarm** (false positive) costs a flat \$4 analyst review.
- A **caught fraud** (true positive) saves `Amount` − \$4.

In [1]:
# --- project-root bootstrap: portable across VS Code / Jupyter / CLI ---
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


## Data dictionary

| Column | Meaning |
|---|---|
| `Time` | Seconds elapsed since the first transaction in the set |
| `Amount` | Transaction amount |
| `V1`–`V28` | PCA-anonymized features (already scaled) |
| `Class` | Target: 1 = fraud, 0 = legitimate |

Source: Kaggle `creditcardfraud` (~284,807 rows, fraud ≈ 0.172%).

In [2]:
import duckdb
CSV = 'data/raw/creditcard.csv'
con = duckdb.connect()
con.execute(f"SELECT * FROM read_csv_auto('{CSV}') LIMIT 5").df()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
bal = con.execute(
    f"SELECT Class, COUNT(*) AS n, "
    f"ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 4) AS pct "
    f"FROM read_csv_auto('{CSV}') GROUP BY Class ORDER BY Class"
).df()
bal

,Class,n,pct
0,0,284315,99.8273
1,1,492,0.1727


In [4]:
exposure = con.execute(
    f"SELECT ROUND(SUM(CASE WHEN Class=1 THEN Amount END), 2) AS fraud_dollars, "
    f"ROUND(AVG(CASE WHEN Class=1 THEN Amount END), 2) AS avg_fraud_amount "
    f"FROM read_csv_auto('{CSV}')"
).df()
exposure

,fraud_dollars,avg_fraud_amount
0,60127.97,122.21


**Next:** `01_ingestion_duckdb.ipynb` — ingest to Parquet and make leakage-safe splits.